# Backend Tester — MTS AI Hub
Тестирует все эндпоинты бэкенда на `http://localhost:8000`

In [3]:
import httpx
import json
import base64
import asyncio
from pathlib import Path

BASE = "http://localhost:8004"
HEADERS = {"Content-Type": "application/json"}

def ok(label, resp):
    status = '✅' if resp.status_code < 400 else '❌'
    print(f"{status} [{resp.status_code}] {label}")
    try:
        data = resp.json()
        print(json.dumps(data, ensure_ascii=False, indent=2)[:800])
    except Exception:
        print(resp.text[:400])
    print()

## 1. Health Check

In [4]:
r = httpx.get(f"{BASE}/v1/health")
ok("Health Check", r)

ConnectError: [Errno 61] Connection refused

## 2. Chat — Text (обычный вопрос)

In [3]:
payload = {
    "model": "gemma-3-27b-it",
    "messages": [{"role": "user", "content": "Привет! Кто ты?"}],
    "stream": False,
    "user": "test_user"
}
r = httpx.post(f"{BASE}/v1/chat/completions", json=payload, timeout=30)
ok("Chat / Text", r)

✅ [200] Chat / Text
{
  "id": "chatcmpl-b9ece0ec1400426ebcb638ec3ea02333",
  "created": 1775932211,
  "model": "gemma-3-27b-it",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Привет! Я — Gemma, большая языковая модель, разработанная Google DeepMind. Я модель с открытым весом, широко доступная для общественного пользования. \n\nЯ могу обрабатывать текст и изображения, и на выходе выдаю только текст. У меня нет доступа к инструментам, информации в режиме реального времени или поиску Google.\n\nМои создатели - команда Gemma.\n",
        "role": "assistant"
      },
      "provider_specific_fields": {
        "stop_reason": 106
      }
    }
  ],
  "usage": {
    "completion_tokens": 82,
    "prompt_tokens": 14,
    "tot



## 3. Chat — Code (роутинг на kodify-2.0)

In [ ]:
payload = {
    "model": "qwen3-coder-480b-a35b",
    "messages": [{"role": "user", "content": "Напиши функцию сортировки пузырьком на Python"}],
    "stream": False,
    "user": "test_user"
}
r = httpx.post(f"{BASE}/v1/chat/completions", json=payload, timeout=30)
ok("Chat / Code routing", r)

## 4. Chat — Streaming (SSE)

In [29]:
payload = {
    "model": "mws-gpt-alpha",
    "messages": [{"role": "user", "content": "Расскажи анекдот"}],
    "stream": True,
    "user": "test_user"
}
print("🔄 Streaming response:")
chunks = []
with httpx.stream("POST", f"{BASE}/v1/chat/completions", json=payload, timeout=60) as resp:
    print(f"Status: {resp.status_code}")
    for line in resp.iter_lines():
        if line.startswith("data: ") and line != "data: [DONE]":
            try:
                chunk = json.loads(line[6:])
                delta = chunk["choices"][0]["delta"].get("content", "")
                if delta:
                    chunks.append(delta)
                    print(delta, end="", flush=True)
            except Exception:
                pass
print(f"\n\n✅ Total chunks: {len(chunks)}")

🔄 Streaming response:
Status: 200
Журналист спросил у старика: "Папа, вы уже жили при царе?"

Старик ответил: "Да, да, я был маленьким мальчиком."

"А как выглядел царь?" - спросил журналист.

"Волосы у него были черные, а борода - белая." - ответил старик.

"Это был царь Иван Грозный?" - спросил журналист.

"Нет, это был царь Феодор Иоаннович." - ответил старик.

"Папа, вы путаете!" - сказал журналист.

"Нет, нет, я помню, Иван Грозный был в красной рубашке, а Феодор Иоаннович в синей." - ответил старик.

✅ Total chunks: 187


## 5. Deep Research

In [13]:
payload = {
    "query": "Что такое квантовые вычисления?",
    "user_id": "test_user"
}
print("🔄 Research streaming:")
with httpx.stream("POST", f"{BASE}/v1/research", json=payload, timeout=100) as resp:
    print(f"Status: {resp.status_code}")
    lines_collected = []
    for line in resp.iter_lines():
        if line:
            lines_collected.append(line)
            print(line[:120])
        # if len(lines_collected) > 10:
        #     print("... (truncated)")
        #     break
print("\n✅ Research streaming OK")

🔄 Research streaming:


ConnectError: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение

## 6. Web Search

In [22]:
payload = {"query": "Python FastAPI tutorial", "max_results": 3}
r = httpx.post(f"{BASE}/v1/tools/web-search", json=payload, timeout=20)
ok("Web Search", r)

ConnectError: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение

## 7. Web Parse

In [7]:
payload = {"url": "https://funpay.com/"}
r = httpx.post(f"{BASE}/v1/tools/web-parse", json=payload, timeout=20)
ok("Web Parse", r)

✅ [200] Web Parse
{
  "url": "https://funpay.com/",
  "text": "FunPay — биржа игровых ценностей\nСреди\nпампищеков\nНовости\nи мемы\n42 803\nотзыва\n0\nпретензий\n11 704\nотзыва\nНовости\nи мемы\nВсе игры по алфавиту\nA\nAbyss of Dungeons\nАккаунты\nДонат\nПредметы\nУслуги\nПрочее\nAcrobat\nАккаунты\nУслуги\nОбучение\nПодписка\nПрочее\nActive Matter\nАккаунты\nКлючи\nУслуги\nTwitch Drops\nПрочее\nAdobe\nАккаунты\nУслуги\nОбучение\nПодписка\nПрочее\nAFK Arena\nАккаунты\nАлмазы\nДонат\nУслуги\nГайды\nAFK Journey\nАккаунты\nКристаллы дракона\nДонат\nПрочее\nAfter Effects\nАккаунты\nУслуги\nОбучение\nПодписка\nШаблоны\nПрочее\nAge of Empires Mobile\nАккаунты\nEmpire Coins\nДонат\nУслуги\nПрочее\nAge of Mythology: Retold\nАккаунты\nКлючи\nОффлайн активации\nGame Pass\nПрочее\nAge of Wonders 4\nАккаунты\nКлючи\nП



## 8. File Upload (TXT)

In [35]:
# Create a temp file to upload
test_txt = Path("/tmp/test_doc.txt")
test_txt.write_text("Это тестовый документ. Python — мощный язык программирования. FastAPI очень быстрый.", encoding="utf-8")

with open(test_txt, "rb") as f:
    r = httpx.post(
        f"{BASE}/v1/files/upload",
        files={"file": ("test_doc.txt", f, "text/plain")},
        data={"user_id": "test_user"},
        timeout=30
    )
ok("File Upload (TXT)", r)

ReadTimeout: timed out

## 9. Memory — Get User Memory

In [8]:
r = httpx.get(f"{BASE}/v1/memory/test_user", timeout=10)
ok("Memory GET test_user", r)

✅ [200] Memory GET test_user
{
  "user_id": "test_user",
  "memories": []
}



## 10. Memory — Search

In [ ]:
payload = {"query": "Python программирование", "top_k": 5}
r = httpx.post(f"{BASE}/v1/memory/test_user/search", json=payload, timeout=10)
ok("Memory Search", r)

## 11. Image Generation

In [37]:
payload = {"prompt": "beautiful sunset over the sea", "width": 512, "height": 512}
r = httpx.post(f"{BASE}/v1/image/generate", json=payload, timeout=60)
ok("Image Generate", r)

✅ [200] Image Generate
{
  "image_b64": null,
  "mime": null,
  "fallback": true,
  "description": "Сервис генерации изображений временно недоступен. Вот текстовое описание запрошенного изображения:\n\n🎨 beautiful sunset over the sea\n\nРазмер: 512×512px, шагов: 20."
}



## 12. PPTX Generation

In [38]:
payload = {
    "topic": "Искусственный интеллект в 2025 году",
    "slides_count": 5,
    "user_id": "test_user"
}
r = httpx.post(f"{BASE}/v1/tools/generate-pptx", json=payload, timeout=60)
if r.status_code == 200:
    out = Path("/tmp/test_output.pptx")
    out.write_bytes(r.content)
    print(f"✅ [200] PPTX saved to {out} ({len(r.content)} bytes)")
else:
    ok("PPTX Generate", r)

❌ [502] PPTX Generate
{
  "detail": "LLM недоступен: "
}



## 13. Voice Message (ASR → LLM → TTS)

In [ ]:
# Requires a real WAV file; here we test the endpoint exists and returns 4xx without audio
r = httpx.post(
    f"{BASE}/v1/voice/message",
    files={"audio": ("test.wav", b"RIFF", "audio/wav")},
    data={"user_id": "test_user"},
    timeout=30
)
status = '✅ endpoint reachable' if r.status_code < 500 else '❌ server error'
print(f"{status} [{r.status_code}] Voice Message")
print(r.text[:300])

## 14. Router Test (детектирование типов)

In [44]:
test_cases = [
    ("Привет, как дела?",                       "text"),
    ("Напиши функцию сортировки на Python",      "code"),
    ("Исследуй тему квантовых вычислений",       "deep_research"),
    ("Зайди на https://example.com и расскажи",  "web_parse"),
    ("Найди новости про ИИ",                     "web_search"),
    ("Нарисуй закат над морем",                  "image_gen"),
]

print("Testing router via /v1/chat/completions with model=auto\n")
for msg, expected in test_cases:
    payload = {
        "model": "auto",
        "messages": [{"role": "user", "content": msg}],
        "stream": False,
        "user": "test_user"
    }
    try:
        r = httpx.post(f"{BASE}/v1/chat/completions", json=payload, timeout=15)
        routed_model = r.json().get("model", "?")
        status = "✅" if r.status_code < 400 else "❌"
        print(f"{status} [{expected:15s}] model={routed_model:30s} | {msg[:50]}")
    except Exception as e:
        print(f"❌ [{expected:15s}] ERROR: {e} | {msg[:50]}")

Testing router via /v1/chat/completions with model=auto

✅ [text           ] model=mws-gpt-alpha                  | Привет, как дела?
❌ [code           ] ERROR: Expecting value: line 1 column 1 (char 0) | Напиши функцию сортировки на Python
❌ [deep_research  ] ERROR: Expecting value: line 1 column 1 (char 0) | Исследуй тему квантовых вычислений
✅ [web_parse      ] model=mws-gpt-alpha                  | Зайди на https://example.com и расскажи
✅ [web_search     ] model=mws-gpt-alpha                  | Найди новости про ИИ
✅ [image_gen      ] model=mws-gpt-alpha                  | Нарисуй закат над морем


## 15. Summary — All Endpoints

In [40]:
endpoints = [
    ("GET",  "/v1/health",                    None),
    ("POST", "/v1/chat/completions",          {"model":"mws-gpt-alpha","messages":[{"role":"user","content":"hi"}],"stream":False}),
    ("POST", "/v1/research",                  {"query":"test","user_id":"u1"}),
    ("POST", "/v1/tools/web-search",          {"query":"test","max_results":1}),
    ("POST", "/v1/tools/web-parse",           {"url":"https://example.com"}),
    ("GET",  "/v1/memory/test_user",          None),
    ("POST", "/v1/memory/test_user/search",   {"query":"test","top_k":3}),
    ("POST", "/v1/image/generate",            {"prompt":"cat"}),
    ("POST", "/v1/vlm/analyze",               {"image_url":"https://example.com/img.jpg","question":"что на фото?"}),
    ("POST", "/v1/tools/generate-pptx",       {"topic":"AI","slides_count":3,"user_id":"u1"}),
]

print(f"{'Method':<6} {'Endpoint':<35} {'Status':>6}  Result")
print("-" * 70)
for method, path, body in endpoints:
    try:
        if method == "GET":
            r = httpx.get(f"{BASE}{path}", timeout=10)
        else:
            r = httpx.post(f"{BASE}{path}", json=body, timeout=15)
        icon = "✅" if r.status_code < 400 else "⚠️ "
        print(f"{method:<6} {path:<35} {r.status_code:>6}  {icon}")
    except Exception as e:
        print(f"{method:<6} {path:<35} {'ERR':>6}  ❌ {str(e)[:40]}")

Method Endpoint                            Status  Result
----------------------------------------------------------------------
GET    /v1/health                             200  ✅
POST   /v1/chat/completions                   ERR  ❌ timed out
POST   /v1/research                           ERR  ❌ peer closed connection without sending c
POST   /v1/tools/web-search                   200  ✅
POST   /v1/tools/web-parse                    200  ✅
GET    /v1/memory/test_user                   200  ✅
POST   /v1/memory/test_user/search            405  ⚠️ 
POST   /v1/image/generate                     200  ✅
POST   /v1/vlm/analyze                        422  ⚠️ 
POST   /v1/tools/generate-pptx                ERR  ❌ timed out
